# CNN vs Vision Transformer on CIFAR-10

**Author:** Aditya Kumar

A PyTorch comparison of a CIFAR-adapted ResNet-18 and a compact Vision Transformer, including learning-rate sensitivity and a residual-connection ablation.

**Recorded-run note:** Outputs below are retained from the original experiment. The validation subset shares the training dataset and therefore receives random cropping and flipping. Test images use deterministic preprocessing. See README.md for limitations and reproduction instructions.


## 1. Setup and Imports

In [1]:
import gc
import json
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
_ = plt.ioff()

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet18

warnings.filterwarnings(
    "ignore",
    message=r"dtype\(\): align should be passed.*",
)

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "font.size": 12,
})

EXP1_HISTORY_FILE = "exp1_histories.json"
EXP2_RESULTS_FILE = "exp2_lr_results.json"
EXP3_HISTORY_FILE = "exp3_ablation_history.json"


def save_json(data, filename):
    """Write JSON atomically to avoid incomplete recovery files."""
    temporary = f"{filename}.tmp"
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2)
    os.replace(temporary, filename)


def load_json(filename):
    """Load a required experiment result file."""
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Required file not found: {filename}")
    with open(filename, "r", encoding="utf-8") as file:
        return json.load(file)


def save_figure(fig, filename):
    """Save and close a figure without triggering inline rendering."""
    try:
        fig.tight_layout()
        fig.savefig(filename, dpi=150, bbox_inches="tight")
    finally:
        plt.close(fig)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | Device: {device}")
if device.type == "cuda":
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu_memory:.1f} GB)")


PyTorch 2.12.1+cu130 | CUDA 13.0 | Device: cuda
GPU: NVIDIA GeForce RTX 5080 Laptop GPU (17.1 GB)


## 2. Reproducibility

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)


## 3. Dataset – CIFAR-10

CIFAR-10 is divided into training, validation, and test sets. Random cropping and horizontal flipping are applied to the training data, and all images are normalised using CIFAR-10 channel statistics.


In [3]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_set = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform,
)
test_set = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform,
)

val_size = 5_000
train_size = len(train_set) - val_size
train_subset, val_subset = torch.utils.data.random_split(
    train_set,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

BATCH_SIZE = 64
NUM_WORKERS = 0
PIN_MEMORY = False

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    test_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print(
    f"Train: {len(train_subset):,} | Validation: {len(val_subset):,} | "
    f"Test: {len(test_set):,} | Batch size: {BATCH_SIZE}"
)
print(f"Classes: {train_set.classes}")


100.0%


Train: 45,000 | Validation: 5,000 | Test: 10,000 | Batch size: 64
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## 4. Model Definitions

### 4.1 ResNet-18 adapted for CIFAR-10
The original ResNet-18 is designed for 224×224 ImageNet images.  
We replace the first convolution with a 3×3 stride-1 kernel and remove the max-pooling layer so that the network works well on 32×32 inputs.

In [4]:
def create_resnet18(num_classes=10):
    """Create a ResNet-18 adapted to 32×32 CIFAR-10 images."""
    model = resnet18(weights=None)
    model.conv1 = nn.Conv2d(
        3, 64, kernel_size=3, stride=1, padding=1, bias=False
    )
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


class BasicBlockNoSkip(nn.Module):
    """Two-convolution block without a residual connection."""

    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_planes,
            planes,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(
            planes,
            planes,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(planes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x)


def create_plain_resnet18(num_classes=10):
    """Create a ResNet-18-shaped network with all skip connections removed."""
    model = resnet18(weights=None)
    model.conv1 = nn.Conv2d(
        3, 64, kernel_size=3, stride=1, padding=1, bias=False
    )
    model.maxpool = nn.Identity()

    def make_layer(in_planes, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []

        for block_stride in strides:
            layers.append(
                BasicBlockNoSkip(in_planes, planes, block_stride)
            )
            in_planes = planes

        return nn.Sequential(*layers)

    model.layer1 = make_layer(64, 64, 2, stride=1)
    model.layer2 = make_layer(64, 128, 2, stride=2)
    model.layer3 = make_layer(128, 256, 2, stride=2)
    model.layer4 = make_layer(256, 512, 2, stride=2)
    model.fc = nn.Linear(512, num_classes)
    return model


### 4.2 Tiny Vision Transformer
A compact ViT designed for 32×32 images (patch size 4 → 8×8 = 64 tokens).

In [5]:
class PatchEmbed(nn.Module):
    def __init__(
        self,
        img_size=32,
        patch_size=4,
        in_chans=3,
        embed_dim=192,
    ):
        super().__init__()
        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )
        self.num_patches = (img_size // patch_size) ** 2

    def forward(self, x):
        x = self.proj(x)
        return x.flatten(2).transpose(1, 2)


class Attention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=3,
        qkv_bias=True,
        attn_drop=0.0,
        proj_drop=0.0,
    ):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        batch_size, tokens, channels = x.shape
        qkv = self.qkv(x).reshape(
            batch_size,
            tokens,
            3,
            self.num_heads,
            channels // self.num_heads,
        )
        q, k, v = qkv.permute(2, 0, 3, 1, 4)

        attention = (q @ k.transpose(-2, -1)) * self.scale
        attention = self.attn_drop(attention.softmax(dim=-1))

        x = (attention @ v).transpose(1, 2).reshape(
            batch_size,
            tokens,
            channels,
        )
        return self.proj_drop(self.proj(x))


class MLP(nn.Module):
    def __init__(self, in_features, hidden_features=None, drop=0.0):
        super().__init__()
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, in_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        return self.drop(self.fc2(x))


class Block(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=2.0,
        drop=0.0,
        attn_drop=0.0,
    ):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(
            dim,
            num_heads=num_heads,
            attn_drop=attn_drop,
            proj_drop=drop,
        )
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(
            dim,
            hidden_features=int(dim * mlp_ratio),
            drop=drop,
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class TinyViT(nn.Module):
    def __init__(
        self,
        img_size=32,
        patch_size=4,
        in_chans=3,
        num_classes=10,
        embed_dim=192,
        depth=6,
        num_heads=3,
        mlp_ratio=2.0,
        drop_rate=0.1,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(
            img_size,
            patch_size,
            in_chans,
            embed_dim,
        )
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(
            torch.zeros(1, num_patches + 1, embed_dim)
        )
        self.pos_drop = nn.Dropout(drop_rate)

        self.blocks = nn.ModuleList([
            Block(
                embed_dim,
                num_heads,
                mlp_ratio,
                drop=drop_rate,
            )
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
        elif isinstance(module, nn.LayerNorm):
            nn.init.constant_(module.bias, 0)
            nn.init.constant_(module.weight, 1.0)

    def forward(self, x):
        batch_size = x.shape[0]
        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = self.pos_drop(x + self.pos_embed)

        for block in self.blocks:
            x = block(x)

        return self.head(self.norm(x)[:, 0])


def count_parameters(model):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


_dummy = torch.randn(2, 3, 32, 32)
_resnet = create_resnet18()
_vit = TinyViT()
_plain = create_plain_resnet18()

with torch.no_grad():
    _resnet_output = _resnet(_dummy).shape
    _vit_output = _vit(_dummy).shape

print(f"ResNet-18 parameters: {count_parameters(_resnet):,}")
print(f"TinyViT parameters: {count_parameters(_vit):,}")
print(f"Plain-ResNet parameters: {count_parameters(_plain):,}")
print(f"Output shapes: ResNet {_resnet_output}, TinyViT {_vit_output}")

del _dummy, _resnet, _vit, _plain, _resnet_output, _vit_output
gc.collect()


ResNet-18 parameters: 11,173,962
TinyViT parameters: 1,806,538
Plain-ResNet parameters: 11,000,138
Output shapes: ResNet torch.Size([2, 10]), TinyViT torch.Size([2, 10])


15

## 5. Training and Evaluation Utilities

In [6]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        predictions = outputs.argmax(dim=1)
        total += targets.size(0)
        correct += predictions.eq(targets).sum().item()

    return running_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item() * inputs.size(0)
        predictions = outputs.argmax(dim=1)
        total += targets.size(0)
        correct += predictions.eq(targets).sum().item()

    return running_loss / total, 100.0 * correct / total


def train_model(
    model,
    train_loader,
    val_loader,
    epochs=10,
    lr=1e-3,
    weight_decay=5e-4,
    device=device,
    model_name="model",
):
    """Train a model and restore its best validation checkpoint."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
    }
    best_acc = 0.0
    best_state = None
    start_time = time.time()

    print(
        f"\n=== Training {model_name} for {epochs} epochs "
        f"(lr={lr}) ==="
    )

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
        )
        val_loss, val_acc = evaluate(
            model,
            val_loader,
            criterion,
            device,
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        scheduler.step()

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | "
            f"Time: {time.time() - epoch_start:.1f}s"
        )

    total_time = time.time() - start_time

    if best_state is not None:
        model.load_state_dict(best_state)

    history["best_val_acc"] = best_acc
    history["total_time"] = total_time

    print(
        f"Finished {model_name} in {total_time / 60:.1f} min. "
        f"Best Val Acc: {best_acc:.2f}%"
    )
    return history, model


## 6. Experiment 1 – Model Comparison (ResNet-18 vs TinyViT)

Both models are trained with the **same** hyper-parameters for a fair comparison.

In [7]:
EPOCHS = 12
LR = 1e-3
WD = 5e-4

set_seed(42)
resnet = create_resnet18().to(device)
hist_resnet, resnet = train_model(
    resnet,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WD,
    model_name="ResNet-18",
)
torch.save(resnet.state_dict(), "resnet18_cifar10.pth")

# Save partial progress before training the second model.
save_json({"hist_resnet": hist_resnet}, EXP1_HISTORY_FILE)

set_seed(42)
vit = TinyViT().to(device)
hist_vit, vit = train_model(
    vit,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WD,
    model_name="TinyViT",
)
torch.save(vit.state_dict(), "tinyvit_cifar10.pth")

save_json(
    {
        "hist_resnet": hist_resnet,
        "hist_vit": hist_vit,
    },
    EXP1_HISTORY_FILE,
)

print("\nExperiment 1 complete.")



=== Training ResNet-18 for 12 epochs (lr=0.001) ===
Epoch 01/12 | Train Loss: 1.3615 Acc: 50.33% | Val Loss: 1.0544 Acc: 62.90% | Time: 18.3s
Epoch 02/12 | Train Loss: 0.8934 Acc: 68.41% | Val Loss: 0.8533 Acc: 70.54% | Time: 18.8s
Epoch 03/12 | Train Loss: 0.7042 Acc: 75.55% | Val Loss: 0.7987 Acc: 73.08% | Time: 18.5s
Epoch 04/12 | Train Loss: 0.5881 Acc: 79.70% | Val Loss: 0.6502 Acc: 77.62% | Time: 19.3s
Epoch 05/12 | Train Loss: 0.5048 Acc: 82.52% | Val Loss: 0.5845 Acc: 80.62% | Time: 19.4s
Epoch 06/12 | Train Loss: 0.4332 Acc: 85.15% | Val Loss: 0.4769 Acc: 83.60% | Time: 19.5s
Epoch 07/12 | Train Loss: 0.3731 Acc: 87.12% | Val Loss: 0.3986 Acc: 86.18% | Time: 19.9s
Epoch 08/12 | Train Loss: 0.3194 Acc: 89.04% | Val Loss: 0.3749 Acc: 87.34% | Time: 19.4s
Epoch 09/12 | Train Loss: 0.2700 Acc: 90.59% | Val Loss: 0.3487 Acc: 87.92% | Time: 19.4s
Epoch 10/12 | Train Loss: 0.2281 Acc: 92.13% | Val Loss: 0.3370 Acc: 88.78% | Time: 19.5s
Epoch 11/12 | Train Loss: 0.1951 Acc: 93.30% | 

In [8]:
criterion = nn.CrossEntropyLoss()

resnet_test_loss, resnet_test_acc = evaluate(
    resnet,
    test_loader,
    criterion,
    device,
)
vit_test_loss, vit_test_acc = evaluate(
    vit,
    test_loader,
    criterion,
    device,
)

save_json(
    {
        "hist_resnet": hist_resnet,
        "hist_vit": hist_vit,
        "resnet_test_loss": resnet_test_loss,
        "resnet_test_acc": resnet_test_acc,
        "vit_test_loss": vit_test_loss,
        "vit_test_acc": vit_test_acc,
    },
    EXP1_HISTORY_FILE,
)

print("\n===== Final Test Results =====")
print(
    f"ResNet-18 | Test Acc: {resnet_test_acc:.2f}% | "
    f"Loss: {resnet_test_loss:.4f}"
)
print(
    f"TinyViT   | Test Acc: {vit_test_acc:.2f}% | "
    f"Loss: {vit_test_loss:.4f}"
)
print(
    f"Parameters | ResNet: {count_parameters(resnet):,} | "
    f"TinyViT: {count_parameters(vit):,}"
)



===== Final Test Results =====
ResNet-18 | Test Acc: 89.35% | Loss: 0.3219
TinyViT   | Test Acc: 66.17% | Loss: 0.9456
Parameters | ResNet: 11,173,962 | TinyViT: 1,806,538


In [9]:
exp1_results = load_json(EXP1_HISTORY_FILE)
hist_resnet = exp1_results["hist_resnet"]
hist_vit = exp1_results["hist_vit"]

resnet_epochs = range(1, len(hist_resnet["train_loss"]) + 1)
vit_epochs = range(1, len(hist_vit["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    resnet_epochs,
    hist_resnet["train_loss"],
    label="ResNet train",
)
axes[0].plot(
    resnet_epochs,
    hist_resnet["val_loss"],
    label="ResNet validation",
)
axes[0].plot(
    vit_epochs,
    hist_vit["train_loss"],
    label="TinyViT train",
)
axes[0].plot(
    vit_epochs,
    hist_vit["val_loss"],
    label="TinyViT validation",
)
axes[0].set(
    title="Loss curves",
    xlabel="Epoch",
    ylabel="Loss",
)
axes[0].legend()

axes[1].plot(
    resnet_epochs,
    hist_resnet["train_acc"],
    label="ResNet train",
)
axes[1].plot(
    resnet_epochs,
    hist_resnet["val_acc"],
    label="ResNet validation",
)
axes[1].plot(
    vit_epochs,
    hist_vit["train_acc"],
    label="TinyViT train",
)
axes[1].plot(
    vit_epochs,
    hist_vit["val_acc"],
    label="TinyViT validation",
)
axes[1].set(
    title="Accuracy curves",
    xlabel="Epoch",
    ylabel="Accuracy (%)",
)
axes[1].legend()

save_figure(fig, "exp1_curves.png")
print("Saved: exp1_curves.png")


Saved: exp1_curves.png


## 7. Experiment 2 – Learning-Rate Study (on ResNet-18)

We train the same architecture with three different learning rates while keeping every other hyper-parameter fixed.

In [10]:
learning_rates = [1e-3, 3e-4, 1e-4]
lr_results = {}

for lr in learning_rates:
    set_seed(42)
    model = create_resnet18().to(device)

    history, model = train_model(
        model,
        train_loader,
        val_loader,
        epochs=EPOCHS,
        lr=lr,
        weight_decay=WD,
        model_name=f"ResNet-18 (lr={lr})",
    )
    test_loss, test_acc = evaluate(
        model,
        test_loader,
        criterion,
        device,
    )

    lr_results[str(lr)] = {
        "history": history,
        "test_acc": test_acc,
        "test_loss": test_loss,
        "best_val_acc": history["best_val_acc"],
    }

    torch.save(
        model.state_dict(),
        f"resnet18_lr_{lr}.pth",
    )
    save_json(lr_results, EXP2_RESULTS_FILE)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n===== Learning-Rate Study Summary =====")
for lr, result in lr_results.items():
    print(
        f"lr={lr:<8} | "
        f"Best Val Acc: {result['best_val_acc']:.2f}% | "
        f"Test Acc: {result['test_acc']:.2f}%"
    )



=== Training ResNet-18 (lr=0.001) for 12 epochs (lr=0.001) ===
Epoch 01/12 | Train Loss: 1.3615 Acc: 50.33% | Val Loss: 1.0544 Acc: 62.90% | Time: 19.8s
Epoch 02/12 | Train Loss: 0.8934 Acc: 68.41% | Val Loss: 0.8533 Acc: 70.54% | Time: 19.5s
Epoch 03/12 | Train Loss: 0.7042 Acc: 75.55% | Val Loss: 0.7987 Acc: 73.08% | Time: 19.6s
Epoch 04/12 | Train Loss: 0.5881 Acc: 79.70% | Val Loss: 0.6502 Acc: 77.62% | Time: 19.8s
Epoch 05/12 | Train Loss: 0.5048 Acc: 82.52% | Val Loss: 0.5845 Acc: 80.62% | Time: 19.7s
Epoch 06/12 | Train Loss: 0.4332 Acc: 85.15% | Val Loss: 0.4769 Acc: 83.60% | Time: 19.8s
Epoch 07/12 | Train Loss: 0.3731 Acc: 87.12% | Val Loss: 0.3986 Acc: 86.18% | Time: 19.9s
Epoch 08/12 | Train Loss: 0.3194 Acc: 89.04% | Val Loss: 0.3749 Acc: 87.34% | Time: 20.0s
Epoch 09/12 | Train Loss: 0.2700 Acc: 90.59% | Val Loss: 0.3487 Acc: 87.92% | Time: 20.0s
Epoch 10/12 | Train Loss: 0.2281 Acc: 92.13% | Val Loss: 0.3370 Acc: 88.78% | Time: 20.1s
Epoch 11/12 | Train Loss: 0.1951 Acc

In [11]:
lr_results = load_json(EXP2_RESULTS_FILE)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lr, result in sorted(
    lr_results.items(),
    key=lambda item: float(item[0]),
    reverse=True,
):
    epochs = range(1, len(result["history"]["val_loss"]) + 1)
    axes[0].plot(
        epochs,
        result["history"]["val_loss"],
        label=f"lr={lr}",
    )
    axes[1].plot(
        epochs,
        result["history"]["val_acc"],
        label=f"lr={lr}",
    )

axes[0].set(
    title="Validation loss by learning rate",
    xlabel="Epoch",
    ylabel="Validation loss",
)
axes[0].legend()

axes[1].set(
    title="Validation accuracy by learning rate",
    xlabel="Epoch",
    ylabel="Validation accuracy (%)",
)
axes[1].legend()

save_figure(fig, "exp2_lr_study.png")
print("Saved: exp2_lr_study.png")


Saved: exp2_lr_study.png


## 8. Experiment 3 – Ablation Study (Residual Connections)

We compare the original ResNet-18 with a **Plain-ResNet-18** in which every residual (skip) connection has been removed.  
This directly tests the importance of residual learning on CIFAR-10.

In [12]:
set_seed(42)
plain = create_plain_resnet18().to(device)

hist_plain, plain = train_model(
    plain,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WD,
    model_name="Plain-ResNet-18 (no residual)",
)

plain_test_loss, plain_test_acc = evaluate(
    plain,
    test_loader,
    criterion,
    device,
)
torch.save(
    plain.state_dict(),
    "plain_resnet18_cifar10.pth",
)

save_json(
    {
        "hist_plain": hist_plain,
        "plain_test_loss": plain_test_loss,
        "plain_test_acc": plain_test_acc,
    },
    EXP3_HISTORY_FILE,
)

resnet_test_acc = load_json(EXP1_HISTORY_FILE)["resnet_test_acc"]

print("\n===== Ablation Results =====")
print(
    f"ResNet-18 (with residual)  | Test Acc: "
    f"{resnet_test_acc:.2f}%"
)
print(
    f"Plain-ResNet (no residual) | Test Acc: "
    f"{plain_test_acc:.2f}%"
)
print(
    f"Difference                 | "
    f"{resnet_test_acc - plain_test_acc:.2f} percentage points"
)



=== Training Plain-ResNet-18 (no residual) for 12 epochs (lr=0.001) ===
Epoch 01/12 | Train Loss: 1.7261 Acc: 34.47% | Val Loss: 1.6716 Acc: 40.16% | Time: 19.8s
Epoch 02/12 | Train Loss: 1.3175 Acc: 51.68% | Val Loss: 1.5308 Acc: 45.70% | Time: 19.9s
Epoch 03/12 | Train Loss: 1.1044 Acc: 60.07% | Val Loss: 1.1732 Acc: 57.80% | Time: 20.0s
Epoch 04/12 | Train Loss: 0.9543 Acc: 66.06% | Val Loss: 0.9448 Acc: 66.38% | Time: 19.9s
Epoch 05/12 | Train Loss: 0.8296 Acc: 70.70% | Val Loss: 0.7911 Acc: 72.08% | Time: 19.9s
Epoch 06/12 | Train Loss: 0.7254 Acc: 74.74% | Val Loss: 0.7269 Acc: 74.52% | Time: 19.8s
Epoch 07/12 | Train Loss: 0.6370 Acc: 77.71% | Val Loss: 0.6372 Acc: 77.42% | Time: 20.1s
Epoch 08/12 | Train Loss: 0.5699 Acc: 80.16% | Val Loss: 0.5469 Acc: 80.70% | Time: 19.7s
Epoch 09/12 | Train Loss: 0.4970 Acc: 82.73% | Val Loss: 0.5435 Acc: 81.18% | Time: 19.6s
Epoch 10/12 | Train Loss: 0.4504 Acc: 84.36% | Val Loss: 0.5101 Acc: 82.68% | Time: 19.7s
Epoch 11/12 | Train Loss: 0

In [13]:
hist_resnet = load_json(EXP1_HISTORY_FILE)["hist_resnet"]
hist_plain = load_json(EXP3_HISTORY_FILE)["hist_plain"]

resnet_epochs = range(1, len(hist_resnet["val_loss"]) + 1)
plain_epochs = range(1, len(hist_plain["val_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    resnet_epochs,
    hist_resnet["val_loss"],
    label="ResNet-18 (residual)",
)
axes[0].plot(
    plain_epochs,
    hist_plain["val_loss"],
    label="Plain-ResNet (no residual)",
)
axes[0].set(
    title="Validation loss: residual ablation",
    xlabel="Epoch",
    ylabel="Validation loss",
)
axes[0].legend()

axes[1].plot(
    resnet_epochs,
    hist_resnet["val_acc"],
    label="ResNet-18 (residual)",
)
axes[1].plot(
    plain_epochs,
    hist_plain["val_acc"],
    label="Plain-ResNet (no residual)",
)
axes[1].set(
    title="Validation accuracy: residual ablation",
    xlabel="Epoch",
    ylabel="Validation accuracy (%)",
)
axes[1].legend()

save_figure(fig, "exp3_ablation.png")
print("Saved: exp3_ablation.png")


Saved: exp3_ablation.png


## 9. Summary Tables

In [14]:
exp1_results = load_json(EXP1_HISTORY_FILE)
exp2_results = load_json(EXP2_RESULTS_FILE)
exp3_results = load_json(EXP3_HISTORY_FILE)

_parameter_models = {
    "ResNet-18": create_resnet18(),
    "TinyViT": TinyViT(),
    "Plain-ResNet (ablation)": create_plain_resnet18(),
}
parameter_counts = {
    name: count_parameters(model)
    for name, model in _parameter_models.items()
}
del _parameter_models
gc.collect()

summary_data = [
    {
        "Model": "ResNet-18",
        "Params": parameter_counts["ResNet-18"],
        "Best Val Acc": exp1_results["hist_resnet"]["best_val_acc"],
        "Test Acc": exp1_results["resnet_test_acc"],
        "Train Time (min)": (
            exp1_results["hist_resnet"]["total_time"] / 60
        ),
    },
    {
        "Model": "TinyViT",
        "Params": parameter_counts["TinyViT"],
        "Best Val Acc": exp1_results["hist_vit"]["best_val_acc"],
        "Test Acc": exp1_results["vit_test_acc"],
        "Train Time (min)": (
            exp1_results["hist_vit"]["total_time"] / 60
        ),
    },
    {
        "Model": "Plain-ResNet (ablation)",
        "Params": parameter_counts["Plain-ResNet (ablation)"],
        "Best Val Acc": exp3_results["hist_plain"]["best_val_acc"],
        "Test Acc": exp3_results["plain_test_acc"],
        "Train Time (min)": (
            exp3_results["hist_plain"]["total_time"] / 60
        ),
    },
]

df_summary = pd.DataFrame(summary_data)
df_summary["Params"] = df_summary["Params"].map("{:,}".format)
df_summary["Best Val Acc"] = df_summary["Best Val Acc"].map(
    "{:.2f}%".format
)
df_summary["Test Acc"] = df_summary["Test Acc"].map(
    "{:.2f}%".format
)
df_summary["Train Time (min)"] = df_summary[
    "Train Time (min)"
].map("{:.1f}".format)

print("===== Overall Summary =====")
display(df_summary)

lr_table = [
    {
        "Learning Rate": float(lr),
        "Best Val Acc": f"{result['best_val_acc']:.2f}%",
        "Test Acc": f"{result['test_acc']:.2f}%",
    }
    for lr, result in sorted(
        exp2_results.items(),
        key=lambda item: float(item[0]),
        reverse=True,
    )
]

print("\n===== Learning-Rate Study =====")
display(pd.DataFrame(lr_table))


===== Overall Summary =====


,Model,Params,Best Val Acc,Test Acc,Train Time (min)
0,ResNet-18,"11,173,962",89.90%,89.35%,3.9
1,TinyViT,"1,806,538",65.82%,66.17%,3.0
2,Plain-ResNet (ablation),"11,000,138",85.04%,84.56%,4.0



===== Learning-Rate Study =====


,Learning Rate,Best Val Acc,Test Acc
0,0.0010,89.90%,89.35%
1,0.0003,88.66%,88.33%
2,0.0001,83.02%,83.35%


## 10. Load Saved Weights & Final Verification

This cell demonstrates that the saved weights can be loaded and used for inference (required by the assignment).

In [15]:
criterion = nn.CrossEntropyLoss()

resnet_loaded = create_resnet18().to(device)
resnet_loaded.load_state_dict(
    torch.load(
        "resnet18_cifar10.pth",
        map_location=device,
        weights_only=True,
    )
)
resnet_loaded.eval()

vit_loaded = TinyViT().to(device)
vit_loaded.load_state_dict(
    torch.load(
        "tinyvit_cifar10.pth",
        map_location=device,
        weights_only=True,
    )
)
vit_loaded.eval()

_, loaded_resnet_acc = evaluate(
    resnet_loaded,
    test_loader,
    criterion,
    device,
)
_, loaded_vit_acc = evaluate(
    vit_loaded,
    test_loader,
    criterion,
    device,
)

print(
    f"Loaded ResNet-18 Test Acc: {loaded_resnet_acc:.2f}%"
)
print(
    f"Loaded TinyViT Test Acc: {loaded_vit_acc:.2f}%"
)
print("Saved model weights loaded and verified successfully.")


Loaded ResNet-18 Test Acc: 89.35%
Loaded TinyViT Test Acc: 66.17%
Saved model weights loaded and verified successfully.
